# Confident Legal Research Application

- Main stakeholder/consumer: Junior Legal Assistant
- Distributed: No


**Goal:Improve quality & speed of case research for junior assistants**

## 1. Baseline System

To create an evaluation process we would need to have some baseline system in place. That is why we are creating a **Naive implementation** with the following initial setup:

- One dense vector embedding (`BAAI/bge-small-en-v1.5`, 384-dim, 512x512 context window)
- Search function with no reranker
- Naive RAG pipeline

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Initialize dataset

We will use the famous CLERC dataset for this tutorial. Two reasons:
1. There exist queries in the dataset that we can use as a baseline for complex law queries 
2. The combination of queries/positive/negative responses provides a good baseline for the *golden set* that could be used for testing

CLERC dataset is considered to be a challanging one especially for the semantic searh, the baseline for NDCG@10 ~ 0.2. Check their paper for [more](https://aclanthology.org/2025.findings-naacl.441.pdf?utm_source=chatgpt.com)

In [2]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [3]:
sample_tuple = clerc_data.sample()

sample_tuple

(Query(query_id='482127', query='held a scheduling conference at which a scheduling order was imposed that set discovery and dispositive motion deadlines, among others. Appellants then moved to stay discovery until a ruling on the motions for summary judgment was issued. On March 30, 2006, the district court issued a minute order striking the pending summary judgment motions, to be reurged consistent with the scheduling order, and finding the motion to stay moot. Appellants argue on appeal that the district court erred in refusing to address their respective qualified immunity motions and that this refusal is immediately appealable. Appellee argues that because the district court has not yet ruled on the issue of qualified immunity, the issue is not ripe for appeal. Our decisions in  REDACTED  and Lowe v. Town of Fairland, Okla., 143 F.3d 1378 (10th Cir.1998), make clear that a district court’s postponement of or failure to rule on a qualified immunity defense is immediately appealable

In [4]:
dict(list(clerc_data.qrels.items())[:2])


{'247774': {'22437426'}, '736597': {'195263'}}

### 1.2 Initialize vectors 

- use `qdrant_client` for storing vectors
- use `fastembed` for the simplest embeddings generation pipelines


We will also store the embeddings locally and try to handle them in parallel.

>NB!: This notebook is optimized to run on macbook devices. Try to adapt embedding config to your specific hardware

In [5]:
from qdrant_client import QdrantClient
import os

qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

In [6]:
# constants
COLLECTION_NAME = 'lawyer_citations'
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SIZE = 384

In [7]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance


config = EmbeddingConfig(
    name='dense_base',
    model_id=DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
    providers=["CoreMLExecutionProvider", "CPUExecutionProvider"],
    parallel=None,
)

In [8]:
from src.indexing import EmbeddingCache
embedding_cache = EmbeddingCache()

In [9]:
from src.indexing import DocumentIndexer

document_vectors = DocumentIndexer(
    client,
    COLLECTION_NAME,
    embeddings=[config],
    cache=embedding_cache
)

In [10]:
document_vectors.ensure_collection()

In [20]:
document_vectors.upload(items=list(clerc_data.corpus.values()), batch_size=128)

embed:dense_base:   0%|          | 0/41167 [00:00<?, ?it/s]

In [11]:
client.get_collection(COLLECTION_NAME).points_count, (client.get_collection(COLLECTION_NAME).points_count or 0) > 0

(41167, True)

### 1.3 Create simple semantic search and RAG pipelines

- `DenseSearcher` is the simplest implementation of the semantic search: no sparse vectors, multivectors, no reranking. Just query API
- `LegalAssistant` - uses `DenseSearcher` for the retrieval and generates the results based on that. We use `anthropic api` with `lite-llm`

In [12]:
from src.search import DenseSearcher
from src.rag import LegalAssistant

qa = clerc_data.sample()
searcher = DenseSearcher(client, COLLECTION_NAME, config)
legal_assistant = LegalAssistant(searcher)


In [ ]:
result = searcher.search(qa[0].query)
result[0].text, qa[1].text

### 1.4 Simple test of the QA tool

In [14]:
answer = legal_assistant.ask("My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems")

print(answer)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Based on the case excerpts provided, none of them directly address the specific legal issue of a landlord seeking payment for property damage (a broken wall) in the absence of a signed hand-in (or check-in/check-out) protocol. The excerpts cover unrelated legal matters such as bankruptcy and post-petition claims, agricultural rent assignments, government contract disputes, emergency rent control regulations, and bankruptcy court jurisdiction. None of these cases involve a landlord-tenant dispute over property damage or the evidentiary significance of a missing hand-in/condition report protocol.

Therefore, the provided excerpts do not contain sufficient information to answer your question about cases involving a landlord's claim for wall damage where no hand-in protocol was signed. You would need to consult additional case law specifically addressing landlord-tenant property damage disputes and the burden of proof in the absence of a move-in/move-out condition report.

Citations:
  [37

### 1.5 Baseline - LGTM dataset 

We can always start with the simplest approach, to understand if we even need to delve deep into the evaluations - a good old **Looks Good to Me (LGTM later)**. Stated simply: we can use either LLM or human to define if the queries are actually making good or no. This is a good starting point in case when decision whether system is working according to requirements or not.

In [15]:
lgtm_dataset = [
    'The plaintiff seeks expectation damages after the defendant failed to deliver goods under a commercial supply agreement. What precedents discuss breach of contract, expectation damages, and the duty to mitigate losses?', # Contract Law
    'A customer was injured on business premises after multiple intervening events contributed to the accident. What cases analyze duty of care, proximate cause, and foreseeability?', # Tort Law
    "My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems", # Tenant Law
]

#### Contract Law

In [16]:
answer = legal_assistant.ask(lgtm_dataset[0])

print(answer)

The excerpts address several relevant legal principles concerning breach of contract, expectation damages, and the duty to mitigate. Here is a summary of the key precedents and rules:

1. Proof of Damages and Reasonable Certainty

Under New York law, once it is established that a breach of contract has caused damages, the plaintiff need only prove the amount of those damages with "reasonable certainty" — not absolute precision. The foundational rule, articulated in Wakeman v. Wheeler & Wilson Mfg. Co., 101 N.Y. 205, 209 (1886), is that "[w]hen it is certain that damages have been caused by a breach of contract, and the only uncertainty is as to their amount, there can rarely be good reason for refusing, on account of such uncertainty, any damages whatever for the breach." Courts have further held that "[a] person violating his contract should not be permitted entirely to escape liability because the amount of the damages which he has caused is uncertain," and that "the wrongdoer must s

### Tort Law 


In [17]:
answer = legal_assistant.ask(lgtm_dataset[1])

print(answer)

Based on the provided excerpts, the following analysis applies to duty of care, proximate cause, and foreseeability in the context of customer injuries on business premises:

1. Duty of Care: Tort liability fundamentally depends on the violation of a duty of care owed to the injured person. The scope of that duty — whether it extends to bystanders, customers, or other classes of plaintiffs — is ordinarily defined by the common law. Legislatures can create new duties of care, but the mere fact that a statute defines due care does not, by itself, create a tort-enforceable duty. Courts have consistently held that a plaintiff must first establish that the defendant owed them a duty before any further negligence analysis can proceed.

2. Proximate Cause: Causation is described as "the most susceptible to summary determination" among all elements necessary to support recovery in a tort action. To demonstrate proximate cause, a plaintiff must show that the defendant, through affirmative actio

In [18]:
answer = legal_assistant.ask(lgtm_dataset[2])

print(answer)

Based solely on the case excerpts provided, none of the retrieved cases directly address the specific legal issue of a landlord seeking payment for property damage (a broken wall) in the absence of a signed hand-in (move-out/check-out) protocol. The excerpts cover unrelated legal matters, including bankruptcy and furniture repossession (doc_id: 9457344), agricultural payment assignments and landlord rent notes (doc_id: 14718586), government contract equitable adjustments (doc_id: 8705202), emergency rent control regulations and overcharges (doc_id: 3706151), and bankruptcy bar orders and asset distribution (doc_id: 4359176). None of these cases involve a landlord's claim for property damage to a rental unit, nor do they address the evidentiary or contractual significance of a missing hand-in or move-out protocol.

Therefore, the provided excerpts do not contain sufficient information to answer your question about cases involving a landlord's claim for wall damage in the absence of a si

### 1.6 Results interpetation 

From the first side the AI generated queries are being passed. However, if you ask stronger LLM model to analyze the actual result it will identify two issues with the results if the `lgtm_dataset[0]` and `lgtm_dataset[1]` queries replies:

**Confidence exceeds evidence** (prompt discipline is broken - section 3 and summary are full of speculation in first query) - grave mistake cause it will confuse the Junior Assistants in the same vein it does us. Law needs not just confidence, but a lot of precision

That is why I urge the practitioners to write some queries without using AI if they want to judge manually (AI generated queries -> stronger AI judgements, human generated queries -> both human and AI judgements can work). For the `lgtm_dataset[2]` we see a clear outlier: `Based on the case excerpts provided, none of them directly address the specific legal issue `. This either means that our query is wrong for this dataset (mostly this question was from personal experience in EU not US), however this does not reflect the result => our naive implementation of search still gave wrong results. This is a clear signal now that evaluation is needed to setup a **baseline/AS IS**

## 2 Basis of the evaluation framework

Even on a single query, the retrieval layer is visibly underperforming — by design. The PoC intentionally omits sparse vectors, reranking, and query rewriting so the failure modes those techniques usually hide surface up front.

What saved this run is the LLM: `claude-sonnet-4.6` is strong enough to notice the retrieved excerpts don't answer the question and refuse to fabricate. That safety net is fragile. With a weaker model, an opaque pipeline, or domain experts who can't audit the citation chain, the same retrieval bug becomes invisible — and the system confidently misleads a junior associate who can't yet tell the difference.

### 2.1 Goal oriented evaluation specification

The evaluation is designed **top-down**: a system that doesn't serve its business goal isn't worth measuring at the component level.

### 2.2 Inputs/Assumptions

The preconditions feeding into the methodology:
- **GOAL:** Confident Legal Research
- **CONSUMER:** Junior associate — limited domain depth, needs guidance and explanations
- **DISTRIBUTED:** No — single goal, single consumer

In [33]:
from IPython.display import HTML

HTML("""
<div style="text-align: center;">
    <img src="diagrams/eval-inputs.svg" width="300">
</div>
""")

### 2.3 Goal Decomposition

**Goal decomposition** bridges the abstract business goal and the concrete metrics that measure it. Before we can quantify, we have to be specific about *what the system is* and *what it must do*.

Two questions to anchor the decomposition:
- **Is this a *business* goal?** Yes — it describes the system's *utility*, not its internals.
- **Can we make *component-level* assumptions?** Yes — the system has two: **searcher** (retrieval, with well-understood mechanics) and **rag** (LLM-driven generation).

Refined business objective:

`Improve quality & speed of case research for junior associates`

Technical commitments already baked into the codebase:

`Confident & fast legal agentic search` *(too vague to test as-is — decomposition follows)*

Decomposition is one of the oldest design principles in software engineering. Here we use it to turn the vague tech goal into the evaluation context:

![System Goal Decomposition](diagrams/system-goal-decomposition.svg)

At the system level, three NFRs apply to every sub-system:
- **Trustworthiness** — can the consumer rely on the output without verification, and fail loudly when it can't?
- **Correctness** — does the output match ground truth for the task it owns?
- **Performance** — is it fast enough to be useful in the consumer's loop?

These stay vague until we map them to individual components. Each sub-system has a different consumer, and the consumer determines what each NFR *operationally means*:

![Component Level Goal Decomposition](diagrams/component-level-goal-decomposition.svg)

**Search** is consumed by RAG (programmatic). Its failure surface is narrow — a bad ranked list, an over-confident empty result, or a slow query. Three flat axes are enough.

**RAG** is consumed by a junior associate (human). Its failure surface is richer — confidently-wrong differs sharply from correctly-refused, and citation grounding, hedging, and clarity are all separately damaging when they fail. So the decomposition goes deeper on the RAG side.

**The depth asymmetry is itself the finding:** it tells the reader where the evaluation effort has to concentrate.

One consequence worth calling out early: dense semantic search has no natural zero — every query embeds *somewhere*, every doc embeds *somewhere*, cosine similarity always produces an ordering. So *no relevant doc in the corpus → empty top-K* isn't emergent behavior; it has to be engineered via a calibrated threshold. That's why **Calibration** sits as the headline trustworthiness bullet on the search side.

Without per-component decomposition, a wrong final answer can't be localized to bad retrieval vs. bad synthesis — and what can't be localized can't be fixed. That's exactly what the current PoC shows: RAG had to reject its own retriever's ranking because there was no earlier signal — no calibrated empty, no per-component metric — that would have told the operator the search side was the problem.

## 3. Direct Mapping of the Non-functional requirement into the metrics

### 3.1 Ranking Quality Mapping


Next part of the tutorial we will focus only on one non-functional requirement that obviously needs some improvement by the simple implementation we have. I am talking about **Ranking quality**, a search component NFR (non-functional requirement, will be used from now on).

What we do is walk through a series of decision questions that help identify the most appropriate metric to use in this situation.

![Evaluation Metric Definition](./diagrams/evaluation-path-to-metric.svg)


The end result is that currently have a clear path why we would measure `NDCG@10` - because:
- We have the golden dataset from `CLERC`
- We want to measure what is the current **Ranking Quality** of the system

We simplified two routes here though: 
- There is no validation of the coverage / leakage / drift for the dataset, that is a separate topic within itself
- We assume that for now it will be enough to ignore the consumer being an LLM instead of a person. Technically this can be a mistake — however, acknowledging that our metric is flawed and doing the best we can with it is better than not measuring at all.

Let us run `NDCG@10` against our dataset and interpret the results

In [19]:
from src.evaluation import evaluate_retrieval

report = evaluate_retrieval(searcher, clerc_data)  # smoke run
print(report)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1384
  recall@10    0.2890
  mrr@10       0.0931


Our `ndcg@10` on a 2000-query slice of CLERC **train** is `0.1384`. This lands in the same neighborhood as the published LegalBERT-DPR result of `0.147` — see other [benchmarks](https://www.emergentmind.com/topics/clerc-dataset).

**Caveat:** the published number is measured on the full CLERC *test* split; we're on a subset of *train*. Treat this as a directional sanity check ("we're in the right ballpark for an off-the-shelf dense encoder on legal text"), not a benchmark reproduction. To make the comparison honest, we'd need to rerun on the full test split.

Two reasons the number is low in absolute terms:
1. The legal dataset is domain specific, highly lexical and has complex queries
2. The dataset we are using is actually a trimmed version of the real one. We might expect lower results on the full corpora

## 4.0 Improvement strategies

### 4.1 Trying to naively use Qdrant skills


We can now use qdrant best practices and ask our LLM to use claude skills to improve the system. Its as easy as prompting into the claude-code the following:
```
Use qdrant skills: https://skills.qdrant.tech/search?query=about%20search%20quality%20improvements to find how to improve the current naive search from @src/search.py
```

![Claude Code Running](./diagrams/improvement-claude-skills.png)




Most probably it will go into one of the following strategies:
- [Use Hybrid Search with BM25](https://qdrant.tech/documentation/search/hybrid-queries/?selector=aHRtbCA%2BIGJvZHkgPiBtYWluID4gc2VjdGlvbiA%2BIGRpdiA%2BIGRpdiA%2BIGRpdjpudGgtb2YtdHlwZSgyKSA%2BIGRpdiA%2BIGRpdjpudGgtb2YtdHlwZSgxKSA%2BIGFydGljbGUgPiBoMjpudGgtb2YtdHlwZSgxKQ%3D%3D&q=Hybrid+Search)
- [Use richer embeddings](https://qdrant.tech/documentation/manage-data/points/)

### 4.2 Hybrid Search as a solution

This time it generated for me Hybrid Search class. Since we plan to also create a new collection we can increase the dense embeddings size as well  

In [ ]:
import pandas as pd 
from fastembed import TextEmbedding

supported_models = (
    pd.DataFrame(TextEmbedding.list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["sources", "model_file", "additional_files"])
    .reset_index(drop=True)
)
supported_models

In [16]:

DENSE_MODEL_V2  = 'BAAI/bge-base-en-v1.5' #  0.320 GB,	768 DIM	
DENSE_SIZE_V2 = 768
COLLECTION_NAME_V2 = 'trec_dl_v2'
SPARSE_MODEL_V2 = 'Qdrant/bm25' 

In [17]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance
from qdrant_client.models import Modifier


configs_v2 = [
    EmbeddingConfig(
        name='dense_base',
        model_id=DENSE_MODEL_V2,
        kind='dense',
        backend='sentence-transformers',
        size=DENSE_SIZE_V2,
        distance=Distance.COSINE,
    ),
    EmbeddingConfig(
        name="bm25",
        model_id=SPARSE_MODEL_V2,
        kind="sparse",
        modifier=Modifier.IDF
    )
]

In [18]:
from src.indexing import DocumentIndexer

document_indexer_v2 = DocumentIndexer(
    client,
    COLLECTION_NAME_V2,
    embeddings=configs_v2,
    cache=embedding_cache
)

In [19]:
document_indexer_v2.ensure_collection()

In [19]:
document_indexer_v2.upload(items=list(clerc_data.corpus.values()), batch_size=256)

In [20]:
from src.search import HybridSearcher

searcher_v2 = HybridSearcher(
    client, 
    COLLECTION_NAME_V2, 
    configs_v2,
)

In [19]:
qa_v2 = clerc_data.sample()

qa_v2

(Query(query_id='694789', query='factors by arriving at a sentence that lies outside the range of reasonable sentences dictated by the facts of the case.” United States v. Irey, 612 F.3d 1160, 1190 (11th Cir.2010) (en banc), cert. denied, — U.S.-, 131 S.Ct. 1813, 179 L.Ed.2d 772 (2011) (quotation omitted). The court commits a clear error of judgment when it imposes a sentence that does not achieve the sentencing goals of § 3553(a), which include reflecting the seriousness of the offense, promoting respect for the law, providing just punishment, and deterring criminal conduct. Id. at 1189; 18 U.S.C. § 3553(a)(2)(A), (B). The court has discretion to impose a sentence upon revocation of supervised release consecutively to other sentences being served by the defendant. See  REDACTED .C. § 3584(a), which permits the court to impose consecutive terms of imprisonment, applies to revocation sentences); see also United States v. Hofierka, 83 F.3d 357, 360-62 (11th Cir.1996) (explaining that the

In [20]:
result = searcher_v2.search(qa_v2[0].query)
result[0].text, qa_v2[1].text

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-12 13:50:05.301 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.


('of the commission of the offense of conviction, rather than the 1998 version, in force at the time of sentencing. However, the 1993 and 1998 versions of the Guidelines are not materially different. The section at issue is § 5G1.3, which allows consecutive sentences in certain circumstances. This guideline remained unchanged between 1993 and 1998; the only change was in the Application Notes. That change, however, is immaterial to Steffen’s case. Both versions of the Application Notes refer to U.S.S.G § 7B1.3, which, like § 5G1.3, remained unchanged between 1993 and 1998. Section 7B1.3(f) specifically provides that a sentence for revocation of probation shall be served consecutively to any sentence currently being served: Any term of imprisonment imposed upon the revocation of probation or supervised release shall be ordered to be served consecutively to any sentence of imprisonment that the defendant is serving, whether or not the sentence of imprisonment being served resulted from t

In [16]:
from src.evaluation import evaluate_retrieval

report_v2 = evaluate_retrieval(searcher_v2, clerc_data)  # smoke run
print(report_v2)

It is top 10 evaluation


search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1760
  recall@10    0.3360
  mrr@10       0.1277


#### Completing the ablation: BM25-only baseline

Before analyzing the hybrid result, an honest comparison note: `Hybrid (bge-base + BM25)` changes **two** variables against the naive baseline — a bigger dense model *and* sparse fusion. Attributing the delta to "hybrid" without isolating the sparse leg is a classic ablation pitfall. The cheap fix: run BM25 alone against the same collection.

In [35]:
from src.search import SparseSearcher
from src.evaluation import evaluate_retrieval

searcher_sparse = SparseSearcher(client, COLLECTION_NAME_V2, configs_v2[1])
report_sparse = evaluate_retrieval(searcher_sparse, clerc_data)
print(report_sparse)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1787
  recall@10    0.3230
  mrr@10       0.1353


### 4.2 Hybrid Search analysis

This is where our theoretical framework stoppes working and reality kicks in. 

We will notice many more times that there are cases when one metric is not enough to tell the whole picture, even if we mapped it one to one in the methodology. If we think carefull about our NFR is to have **good ranking quality**. So far we have still do not know how good is *ndcg@10* - it became better but its still much to be desired. 


This means that there is something else we might have missed. This requires us to check other metrices. Always prefer standard metrices to which you know what it represents - we are flipping the script here and going bottom-up. Thus the most default metrices we can have at this level:
- recall@10
- mrr@10 

Notice that recall@10 is way higher than the ndcg@10 which means we can create a hypothesis: ranking is suboptimal, not the results. 

To check it more, lets get bigger sample of `metric@100`

In [14]:
from src.evaluation import evaluate_retrieval, RetrievalMetric

report_v2_100 = evaluate_retrieval(
    searcher_v2, 
    clerc_data,
    top_k=100,
    metrics=[
        RetrievalMetric.RECALL_100,
        RetrievalMetric.NDCG_100, 
        RetrievalMetric.MRR_100,
    ],
)  # smoke run
print(report_v2_100)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-100):
  recall@100   0.9765
  ndcg@100     0.3120
  mrr@100      0.1546


### 4.3 Statistical significance

Lets check now how relatively well does our new solutions? Using naive approach we get the following:

In [24]:
ndcg_10_naive =  0.1384
ndcg_10_hybrid = 0.1760
percent = (1 - (ndcg_10_naive/ndcg_10_hybrid)) * 100

f'{percent:0.2f}%'

'21.36%'

Impressive isn't it? As with everything in computer science we have to go deeper and ask: is it real though? 

Let's say that now because we have a new embedding, we get a certified win for just one query where NDCG goes from 0.05 to 0.95. However altogether we have thousands of queries. The aggregate still shifts up - but attributing that shift to system change is wrong. Aggregate-only reports can't distinguish many small real improvements from one lucky query.   


So next what we are doing is called `paired testing` which allows us to see the real significance change. The claim is that because each query is scored under both systems that means they are comparable as pairs.  

##### Deep dive into the difference between two results

 How much better, with what confidence, and at what significance `HybridSearch` beast `DenseSearch` for 2000 queries? This test is formally called [Wilcoxon](https://en.wikipedia.org/wiki/Wilcoxon_signed-rank_test)

In [11]:
from src.evaluation import compare_pair, RetrievalMetric

report = compare_pair(
    { 'dense': searcher, 'hybrid': searcher_v2 },
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10
)

report

NameError: name 'searcher' is not defined

In [38]:
print(report)

hybrid vs dense (2000 queries, top-10, 95% CI):
  NDCG@10:    Δ = +0.038   95% CI [+0.027, +0.049]   p = 1.4e-10   hybrid>dense: 22.9% | dense>hybrid: 16.0% | ties: 61.1%
  Recall@10:  Δ = +0.048   95% CI [+0.027, +0.069]   p = 8.0e-06   hybrid>dense: 14.0% | dense>hybrid: 9.2% | ties: 76.9%
  MRR@10:     Δ = +0.035   95% CI [+0.025, +0.045]   p = 1.1e-09   hybrid>dense: 22.9% | dense>hybrid: 16.0% | ties: 61.1%


A more interesting comparison would be to do between two almost identical results of the 
`HybridSearch` againts `SparseSearch` for 2000 queries

In [39]:
from src.evaluation import compare_pair, RetrievalMetric

report = compare_pair(
    {'bm25-only': searcher_sparse, 'hybrid': searcher_v2},
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10,
)


search:bm25-only:   0%|          | 0/2000 [00:00<?, ?it/s]

search:hybrid:   0%|          | 0/2000 [00:00<?, ?it/s]

In [40]:
print(report)

hybrid vs bm25-only (2000 queries, top-10, 95% CI):
  NDCG@10:    Δ = -0.003   95% CI [-0.011, +0.005]   p = 8.5e-03   hybrid>bm25-only: 12.1% | bm25-only>hybrid: 21.2% | ties: 66.7%
  Recall@10:  Δ = +0.013   95% CI [-0.003, +0.029]   p = 1.1e-01   hybrid>bm25-only: 7.1% | bm25-only>hybrid: 5.8% | ties: 87.1%
  MRR@10:     Δ = -0.009   95% CI [-0.016, -0.001]   p = 2.1e-05   hybrid>bm25-only: 12.1% | bm25-only>hybrid: 21.2% | ties: 66.7%


#### Reading the numbers

The improvement is real. The three headline numbers say so:
- **Δ = +0.038** — mean NDCG@10 improvement per query. This is the primary quantity.
- **95% CI [+0.027, +0.049]** — tight, does not cross zero. Not a fluke.
- **p = 1.4e-10** — at N=2000, significance is *cheap*: any real effect gives tiny p-values.
  Significance is a *floor* here, not a *win*.

**Hybrid > Dense: 22.9% | Dense > Hybrid: 16.0% | ties: 61.1%.**

The win/loss/tie split is where the interpretation actually lives:

- On **61% of queries fusion changes nothing** in the top-10. The whole aggregate gain is generated inside the 39% contested slice, so the local effect (~0.10 NDCG per contested query) is much larger than the average suggests. Aggregate deltas hide where the gain lives.
- Hybrid wins the contested queries roughly 3:2 — but **dense alone is strictly better on 16% of queries**. A cascade or router approach might beat either system alone: if we can predict where each side wins, that 16% is recoverable. (The oracle upper bound is one line: score each query with the better of the two systems.)
- The recall row ties far more often (76.9%) than NDCG (61.1%): fusion mostly *reorders* the documents both systems already find, and only sometimes surfaces new ones.

### 4.4 Coming up to the improvement framework 

Formally we already got the result, and came up naturally to another looped methodology which we call integration layer (since it can be easily integrated into out Continous Integration pipeline to serve as both safeguard and addressable goal). Here is a more detailed generalized version of the improvement loop we are currently in: 


![From Evaluation to Improvements](./diagrams/from-evaluation-to-integration.svg)

### 4.5 Do we want to improve more?

§4.3 answered is the delta real? — the inner-loop gate of the diagram above. §4.5 asks a different question: is the current state good enough, or do we open another improvement session? That's the outer-loop gate, and it lives outside every metric on the dashboard.

Sometimes and this is this time the simple solutions really work. However, we should be aware that for the next problem of having a 
disperancy between `Hybrid:NDCG@10` and `Hybrid:NDCG@100` in order of almost 80% - we cannot fully rely on the solutions that AI will give us. LLMs will just take the easiest route of selecting the **reranking** strategy. Now, according to the common consensus - reranking is the most powerful tool (recovering up to [40% of precision](https://app.ailog.fr/en/blog/guides/reranking)) to use for improving search. 

Two branches from the satisfaction gate:

- Satisfied → move to the next NFR or the next component.
- Not satisfied → write a new improvement objective, not a new technique. Rerun the loop of improvement checking on the new model:

For this article, we're not satisfied — for a specific reason. Hybrid:NDCG@10 and Hybrid:NDCG@100 diverge by ~80%: the relevant document is often in the top-100 but not in the top-10. That's a ranking problem, not a recall problem, and it gives us a concrete objective — close the @10/@100 gap — before we name any technique.

An objective thus becomes: reach acceptable (>60% NDCG) based on the NDCG@100 being close to 99% -> capacity is there, ranking performance needs to be worked on.

The hypotheses split cleanly across two subcomponents:
- Representation side 
- Retrieval side 

#### Representation side 
Either the vector space itself isn't discriminative enough at the top of the ranking, or the encoder never sees enough of the passage to build a rich vector in the first place. Two hypotheses live here:
- Chunking — passages exceed the 512-token context of bge-base; the encoder is silently truncating, so more space would give it room to capture the full complexity of the corpus.
- Legal-tuned encoder — a domain-scoped model captures features (statute references, citation patterns, legal-specific semantics) that a general-purpose encoder flattens.

#### Retrieval side 

The representation is fine; what's broken is the ordering given the representation. The candidate here is reranking, and it further splits by architecture:

- Cross-encoder — full query-document attention, highest precision, slowest.
- Late-interaction (e.g. ColBERT) — token-level scoring at index time, faster than cross-encoder, richer than a bi-encoder.


#### Loop 1: Chunking 

Let's start with the hypothesis that `CLERC` has huge legal texts that small encoders cannot simply represent it well. For this will we analyze text corpora (the one we have) if it actually can be tokenized by the `BGE` encoder  

In [32]:
from src.corpus_analysis import analyze_corpus

stats = analyze_corpus(list(clerc_data.corpus.values()))

print(stats)

tokenize:   0%|          | 0/42 [00:00<?, ?it/s]

Corpus stats (41,167 docs, tokenizer=BAAI/bge-base-en-v1.5):
  Total tokens:      21,608,613
  Mean tokens/doc:   524.9
  Length quantiles:  p50=521  p90=600  p95=627  p99=706  max=1520
  Over model max (512 tok): 55.7% of docs
  Truncation info loss: 5.7% of total tokens discarded
  Length distribution:
    >  128 tok: 100.0%
    >  256 tok: 100.0%
    >  512 tok:  55.7%
    > 1024 tok:   0.0%
    > 2048 tok:   0.0%


In [33]:
if stats.needs_chunking():
    print('-> chunking recommended')

-> chunking recommended


Even though the code says that chunking is needed, we always need to analyze the signals first and foremost. The stats themselves give us clearer picture: only 5.7% of total tokens are discarded. Not worth investing time. Especially since each addition to the initial solution comes with certain tradeoffs. If on the other side of the tradeoff we do not get any benefit, we should better skip. In case of chunking for this case study, it comes with:

##### Downsides
- Optimal chunking is contextual, you have to split based on some assumption about the text - requires deep dive into corpora
- Chunking can increase disk space `2x` or even `3x` depending on chunking strategy
- Chunking causes duplication of the results -> means harder query result filtering and post processing

##### Upsides (in this case study)
- 5% improvement in text representation


Clearly a pass

#### Loop 2: Finetuned encoder model

We are testing our assumption that bigger and more specialized encoder could actually improve our system and NDCG metric.  

Let us use [fine-tuned modernbert model](https://free.law/2025/03/11/semantic-search/) available on [hugging face](https://huggingface.co/freelawproject/modernbert-embed-base_finetune_512).


We are doing same steps as before: initializing new collection and adding new embeddings to it. 

In [21]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance
from qdrant_client.models import Modifier


DENSE_MODEL_V3  = 'Free-Law-Project/modernbert-embed-base_finetune_512' #  0.320 GB,	768 DIM	
DENSE_SIZE_V3 = 768
COLLECTION_NAME_V3 = 'trec_dl_v3'
SPARSE_MODEL_V3 = 'Qdrant/bm25' 


configs_v3 = [
    EmbeddingConfig(
        name='dense_base',
        model_id=DENSE_MODEL_V3,
        kind='dense',
        backend='sentence-transformers',
        size=DENSE_SIZE_V3,
        distance=Distance.COSINE,
        device='mps'
    ),
    EmbeddingConfig(
        name="bm25",
        model_id=SPARSE_MODEL_V3,
        kind="sparse",
        modifier=Modifier.IDF
    )
]

In [22]:
from src.indexing import DocumentIndexer

document_indexer_v3 = DocumentIndexer(
    client,
    COLLECTION_NAME_V3,
    embeddings=configs_v3,
    cache=embedding_cache
)

document_indexer_v3

In [23]:
document_indexer_v3.ensure_collection()

In [18]:
document_indexer_v3.upload(items=list(clerc_data.corpus.values()), batch_size=32)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Batches:   0%|          | 0/1287 [00:00<?, ?it/s]

Evaluate new indexer

In [24]:
from src.search import HybridSearcher

searcher_v3 = HybridSearcher(
    client, 
    COLLECTION_NAME_V3, 
    configs_v3,
)

In [22]:
from src.evaluation import evaluate_retrieval, RetrievalMetric

report_v3_10 = evaluate_retrieval(
    searcher_v3, 
    clerc_data,
    top_k=10,
    metrics=[
        RetrievalMetric.RECALL_10,
        RetrievalMetric.NDCG_10, 
        RetrievalMetric.MRR_10,
    ],
)  # smoke run
print(report_v3_10)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  recall@10    0.3685
  ndcg@10      0.1924
  mrr@10       0.1394


Overall the results are the following:

| Metric    | Naive (bge-small) | BM25 only | Hybrid (bge-base + BM25) | Legal ModernBERT |
| --------- | ----------------: | --------: | -----------------------: | ---------------: |
| ndcg@10   |            0.1384 |    0.1787 |                    0.1755 |           0.1924 |
| recall@10 |            0.2890 |    0.3360 |                    0.3365 |           0.3685 |
| mrr@10    |            0.0931 |    0.1277 |                    0.1269 |           0.1394 |

**The ablation column changes the §4.2 story.** BM25 alone matches the full hybrid (0.1787 vs 0.1755 ndcg@10): the bge-base dense leg contributes approximately nothing, and the "hybrid gain" over the naive baseline was the sparse leg all along — the two-variable confound flagged above, caught by a one-cell ablation. The paired test confirms it: NDCG Δ = −0.003 (p = .013), MRR Δ = −0.007 (p = 4e-05), Recall Δ = +0.013 (n.s.) — and the win/loss/tie split reads **bm25>hybrid: 23.0% | hybrid>bm25: 12.0% | ties: 65.0%**. BM25 wins the contested queries almost 2:1, yet the mean delta is ~zero — which is only possible if hybrid's rare wins are about twice as big as its frequent losses. That is exactly the profile of the dense leg here: it occasionally *rescues* a query BM25 can't touch (no lexical overlap with the gold passage — hence the one positive number, recall) while paying a small ranking tax everywhere else. The domain verdict is hard to miss: **this task is keyword-driven, and the sparse leg is the load-bearing component** — a thread §4.10 picks up and explains. Keep this in mind for §4.10: a dataset whose labels reward exact matching is exactly where lexical retrieval carries a weak dense encoder.

We can see gradual improvement in the results for `Legal ModernBERT` of around ~10%. It's good and signifies that we are on the right track — however, not nearly enough to justify the slowness that `Legal ModernBERT` brings with it at retrieval (on MacBook machines at least).

Thus, we can try to address the elephant in the room by the name of `reranker`.

### 4.6 Loop 3: Rerankers — cross-encoder and late interaction

§4.2 left us a concrete objective: close the @10/@100 gap — the relevant passage is usually *in* the pool, ranked too low. The consensus tool for exactly this is a reranker. Two architectures, two experiments:

- **Cross-encoder** — full query-document attention, highest precision, slowest.
- **Late interaction (ColBERT)** — token-level scoring, cheaper than a cross-encoder, richer than a bi-encoder.

In [29]:
from src.rerank import Reranker, CrossEncoderStrategy
from src.evaluation import evaluate_retrieval

searcher_ce = Reranker(
    retriever=searcher_v2,
    strategy=CrossEncoderStrategy(device='mps', model_id="Xenova/ms-marco-MiniLM-L-6-v2"),
    multiplier=10,
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
report_ce = evaluate_retrieval(searcher_ce, clerc_data)
print(report_ce)

Result: ndcg@10 ≈ ⟨fill⟩ — **worse than the un-reranked hybrid it sits on.** Hold that thought and try the second architecture before drawing conclusions.

#### Late interaction, server-side

For ColBERT we index a third, multi-vector slot next to a stronger dense model (mxbai-large) and SPLADE, on a cloud cluster with server-side inference. The rerank then happens inside a single Qdrant Query API call: dense + sparse prefetches fused with RRF, ColBERT MaxSim rescoring the fused pool.

In [30]:
import os
from qdrant_client import QdrantClient

inference_client = QdrantClient(
    url=os.getenv('QDRANT_INF_URL'),
    api_key=os.getenv('QDRANT_INF_API_KEY'),
    cloud_inference=True,
    timeout=300,
)

In [32]:
from qdrant_client import models
from qdrant_client.models import Distance, Modifier
from src.indexing import EmbeddingConfig, DocumentIndexer

COLLECTION_NAME_V5 = 'clerc_v5'
configs_v5 = [
    EmbeddingConfig(
        name='dense',
        model_id='mixedbread-ai/mxbai-embed-large-v1',
        kind='dense',
        size=1024,
        distance=Distance.COSINE,
    ),
    EmbeddingConfig(
        name='splade',
        model_id='prithivida/splade_pp_en_v1',
        kind='sparse',
        modifier=Modifier.IDF,
    ),
    EmbeddingConfig(
        name='colbert',
        model_id='answerdotai/answerai-colbert-small-v1',
        kind='late_interaction',
        size=96,
        distance=Distance.COSINE,
    ),
]

indexer_v5 = DocumentIndexer(inference_client, COLLECTION_NAME_V5, configs_v5)
indexer_v5.ensure_collection()
# rerank-only slot: disable HNSW graph on colbert, it is never searched directly
inference_client.update_collection(
    COLLECTION_NAME_V5,
    vectors_config={"colbert": models.VectorParamsDiff(hnsw_config=models.HnswConfigDiff(m=0))},
)

True

In [ ]:
indexer_v5.upload(list(clerc_data.corpus.values()), 64, 5, True)

In [33]:
from src.search import HybridRerankSearcher

searcher_colbert = HybridRerankSearcher(
    inference_client,
    COLLECTION_NAME_V5,
    prefetch=[configs_v5[0], configs_v5[1]],   # dense + splade, fused with RRF
    rerank=configs_v5[2],                      # colbert rescores the fused pool only
    prefetch_multiplier=25,
)

In [ ]:
from src.evaluation import evaluate_retrieval

report_colbert = evaluate_retrieval(searcher_colbert, clerc_data, limit=100)
print(report_colbert)

#### Loop 3 verdict — and a hypothesis to carry forward

**Both rerankers degrade the ranking they were given.** That's not "small improvement" — it's a signal we don't understand something.

First hypothesis: **our queries are too big for small encoders.** MiniLM sees 512 tokens *for the query+passage pair combined*; CLERC queries alone often exceed that, so the model may be scoring truncated noise. Testable prediction: a reranker with a much larger context window should recover the gain. We'll test it in §4.8 — but first, a cheaper suspicion about the data itself.

### 4.7 Loop 4: Know your data!

What does our data tell us about itself? The corpus is suspiciously uniform — before hypothesizing about models again, check the passages we are actually ranking.

In [34]:
import numpy as np

word_counts = np.array([len(p.text.split()) for p in clerc_data.corpus.values()])
print(
    f"words/passage: min={word_counts.min()}  median={int(np.median(word_counts))}  max={word_counts.max()}\n"
    f"passages at exactly {word_counts.max()} words: {(word_counts == word_counts.max()).mean():.1%}\n"
    f"standard deviation: {np.std(word_counts):.1f}"
)

words/passage: min=153  median=350  max=350
passages at exactly 350 words: 98.0%
standard deviation: 11.0


The distribution is a wall at 350 words — 98% of passages sit exactly at the limit. That's the signature of a **fixed-size chunker**, not of natural documents. The CLERC repo confirms it: it ships full documents (*dids*), ≤350-word chunks (*pids*), and a `mapping.pid2did.tsv` connecting them. The `docid`s we have been treating as documents are **passage ids**.

This matters because our qrels are passage-level. New hypothesis: **we might have positive siblings in our top-K that the evaluation just doesn't account for** — the retriever finds the right *case* but a different *slice* of it, and scores zero. That would neatly explain why rerankers (good at document-level relevance) look worse under passage-level labels.

In [ ]:
from src.doc_mapping import load_pid2did, analyze_siblings

pid2did = load_pid2did(clerc_data.corpus)
sibling_stats = analyze_siblings(clerc_data, pid2did)
print(sibling_stats)

Pinned numbers: our 41,167 "documents" are passages from **23,221 cases**; **21.8%** of queries have siblings of their positive in the pool; **12.7%** even have passages of the correct case mined as their own *hard negatives*. The hypothesis is plausible. But plausible is not quantified — measure how often "right case, wrong slice" actually happens in retrieval output:

In [ ]:
from src.evaluation import diagnose_doc_level

diagnosis = diagnose_doc_level(searcher_v2, clerc_data, pid2did)
print(diagnosis)

#### Loop 4 verdict

**Sibling-only outcomes: 1.1% of queries.** Full document miss: **65.2%.** Even granting full doc-level credit (mean doc precision@10 = 0.051), almost nothing changes. The labels are fine; the retrieval is missing the *case*, not the slice.

**Wrong path — hypothesis killed for the price of one diagnostic run.** That's the loop working: cheap falsification before investment.

### 4.8 Loop 5: Max out the representation

Back to the loop-3 hypothesis: *queries are too big for small encoders*. If capacity and context window are the bottleneck, a frontier stack should show it:

- **Qwen3-Embedding-8B** (32K context, 4096-dim) via OpenRouter for the dense slot — no truncation of long legal passages;
- BM25 for the sparse slot, fused with RRF;
- and for the rerank leg, **Cohere Rerank 4 Pro** (32K context, SOTA) via OpenRouter's `/rerank` endpoint.

If the hypothesis is right, both should clearly beat their small-model counterparts.

In [35]:
import os
from qdrant_client.models import Distance, Modifier
from src.indexing import EmbeddingConfig, DocumentIndexer

COLLECTION_NAME_CLOUD = 'clerc_cloud'

configs_cloud = [
    EmbeddingConfig(
        name='dense',
        model_id='qwen/qwen3-embedding-8b',
        kind='dense',
        backend='openrouter',
        parallel=16,
        size=4096,
        distance=Distance.COSINE,
        doc_options={
            "openrouter-api-key": os.getenv("OPEN_ROUTER_API_KEY"),
            'dimensions': 4096,
            "provider": {"sort": "latency"},
        },
        query_prompt=(
            "Instruct: Given a passage from a legal opinion with a citation removed, "
            "retrieve the cited case passage.\nQuery: "
        ),
    ),
    EmbeddingConfig(
        name='sparse',
        model_id='qdrant/bm25',
        kind='sparse',
        modifier=Modifier.IDF,
    ),
]

In [36]:
indexer_cloud = DocumentIndexer(
    inference_client, COLLECTION_NAME_CLOUD, embeddings=configs_cloud, cache=embedding_cache
)
indexer_cloud.ensure_collection()

In [ ]:
indexer_cloud.upload(items=list(clerc_data.corpus.values()), batch_size=32, skip_existing=True, parallel=10)

In [37]:
from src.search import DenseSearcher, HybridSearcher
from src.evaluation import evaluate_retrieval

searcher_qwen_dense = DenseSearcher(inference_client, COLLECTION_NAME_CLOUD, configs_cloud[0])
searcher_qwen_hybrid = HybridSearcher(inference_client, COLLECTION_NAME_CLOUD, configs_cloud)

# batch-precompute query embeddings; live queries would hit the API one by one
searcher_qwen_hybrid.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)

openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:

print(evaluate_retrieval(searcher_qwen_dense, clerc_data))

In [ ]:
print(evaluate_retrieval(searcher_qwen_hybrid, clerc_data))

In [ ]:
from src.evaluation import compare_pair, RetrievalMetric

report = compare_pair(
    {'qwen-dense': searcher_qwen_dense, 'qwen-hybrid': searcher_qwen_hybrid},
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10,
)
print(report)

Pinned results: dense **0.1978**, hybrid **0.2064** ndcg@10 — our best so far (published LegalBERT-DPR reference: 0.147), and the paired test confirms the sparse leg still adds real signal on top of an 8B embedder (RRF acquitted). recall@100 sits around **0.88**: the capacity *is* there. Now the decisive experiment — the 32K-context reranker:

In [38]:
from src.rerank import OpenRouterRerankStrategy, Reranker
from src.evaluation import evaluate_retrieval

searcher_cohere = Reranker(
    retriever=searcher_qwen_hybrid,
    strategy=OpenRouterRerankStrategy(api_key=os.getenv("OPEN_ROUTER_API_KEY")),
    multiplier=10,
)


In [ ]:

# 200-query smoke run (~200 rerank searches ≈ $0.50); drop `limit` for the full run
print(evaluate_retrieval(searcher_cohere, clerc_data, limit=200))

#### Loop 5 verdict — the hypothesis dies

Cohere Rerank 4 Pro: 32K window — **no truncation** — SOTA reranker. Result: ndcg@10 ≈ **0.13**, and recall@10 drops to **0.28**, *below the 0.40 of the first stage it reranked*.

The loop-3 hypothesis is **falsified**: the failure was never context length. The reranker isn't failing to help — it is actively demoting gold passages. When two architecturally different rerankers fail in the same direction, the problem isn't the models. It's our understanding of the task. This is the textbook case where **rerankers harm, not help.**

### 4.9 Loop 6: The entity hypothesis

One more technique-level idea before questioning the task itself. Legal text is full of hard identifiers — statute cites (`17 U.S.C. § 512(k)`), reporter cites (`958 F.2d 15`), case names (`Doe v. Roe`). Topically-equivalent distractors *shouldn't* share the query's exact identifiers, so an entity-overlap reranker might separate gold where semantic scoring can't.

Discipline first: before building anything, run a **zero-cost diagnostic** — rank each query's gold passage among its own mined hard negatives by IDF-weighted entity overlap. If gold doesn't separate, the idea dies for free.

In [39]:
from src.entity_analysis import analyze_entity_signal

entity_stats = analyze_entity_signal(clerc_data)
print(entity_stats)

entity-rank:   0%|          | 0/2000 [00:00<?, ?it/s]

Entity signal (2,000 queries, gold ranked among own hard negatives):
  Queries with >=1 entity:      1,842 (92.1%), median 6 entities/query
  Median pool size (gold+negs): 21
  Gold w/ zero entity overlap:  49.1%
  Mean relative rank of gold:   0.601 (random = 0.500)
  Gold in top-1 by entities:    4.2%
  Gold in top-3 by entities:    11.8%
  Gold in top-5 by entities:    17.2%
  Random top-1 baseline:        4.8%


#### Loop 6 verdict

| Metric | Entity signal | Random baseline (pool ≈ 21) |
|---|---|---|
| Gold top-1 | 4.1% | **4.8%** |
| Gold top-3 | 11.8% | **~14.3%** |
| Gold top-5 | 17.2% | **~23.8%** |
| Mean relative rank | 0.601 | **0.500** |

Below random on **every** row — and **49.1%** of gold passages share *zero* entities with their query. This is an **anti-signal**: the identifiers a citing court mentions are its *other* citations, which the distractors (cases from the same doctrinal chain) match better than the source does. Killed for the price of a 3-second script — and no implementation was ever built.

#### Loop 7: Trying out n-grams hypothesis

In [40]:
from src.ngram_analysis import analyze_ngram_signal


report = analyze_ngram_signal(clerc_data, max_n=5)

corpus df:   0%|          | 0/41167 [00:00<?, ?it/s]

ngram-rank:   0%|          | 0/2000 [00:00<?, ?it/s]

In [41]:
print(report)

N-gram signal (2,000 queries, median pool 21, BM25 k1=1.5 b=0.75, gold ranked among own hard negatives):
  arm       stratum     queries  rel.rank   top-1   top-3   top-5  zero-ovl
  unigram   all           2,000     0.666    8.6%   15.2%   21.7%      0.0%
  unigram   quoted        1,458     0.662    9.1%   16.0%   22.6%      0.0%
  unigram   unquoted        542     0.676    7.2%   12.9%   19.2%      0.0%
  uni+2gram all           2,000     0.628    9.3%   16.0%   24.5%      0.0%
  uni+2gram quoted        1,458     0.625   10.2%   16.8%   25.2%      0.0%
  uni+2gram unquoted        542     0.635    7.0%   13.8%   22.7%      0.0%
  uni+3gram all           2,000     0.610    9.2%   16.4%   25.8%      0.0%
  uni+3gram quoted        1,458     0.607   10.1%   16.9%   26.0%      0.0%
  uni+3gram unquoted        542     0.616    7.0%   15.1%   25.1%      0.0%
  uni+4gram all           2,000     0.600    9.3%   16.9%   26.2%      0.0%
  uni+4gram quoted        1,458     0.597   10.1%   17.3%  

### 4.10 Know your data 2.0 — reading the dataset's paper

Six loops in, every technique-level hypothesis is dead. Time to interrogate the task definition itself — starting with the dataset's own paper ([CLERC, NAACL'25 Findings](https://aclanthology.org/2025.findings-naacl.441.pdf)). It confirms everything the loops surfaced, independently:

- Their own reranker table shows cross-encoder reranking **degrades** performance on CLERC;
- Zero-shot **BM25 (48.3% R@1k) beats every dense model** they tested (ColBERTv2: 17.6%);
- **300-word queries are optimal** — longer ones add "distracting contextual information about non-central citations";
- Queries formally split into **direct** (contain a quote of the cited case) vs **indirect** — yet `grep` finds only 16% of direct quotes, because ≥25% of quotes carry bracket/punctuation alterations.

#### The label is causal, not semantic

Gold is determined by which case the judge *actually cited* — a historical fact — while the retrieved pool is full of equally-relevant candidates. Query `416202` makes it visible. The query text reads:

> *"Absolute legislative immunity is a doctrine that protects individual legislators from liability for their legislative activities. That doctrine does not protect the governing bodies on which they serve."* **[citation removed]**

The gold passage contains **both sentences verbatim** (with the citations interleaved). The evidence that identifies the source is a *copied span* — invisible to embeddings, which pool 350 words into one vector, and to relevance rerankers, which score topicality. No amount of technique-shopping at the ranking layer could have told us this; only interrogating the data did.

### 4.11 Loop 7: The bigram prediction

Every previous loop *shopped for a technique* and tested it. This loop is different: it is the first hypothesis **derived from the data insight** in §4.10. If gold passages are identified by *copied spans*, then contiguity must discriminate where vocabulary can't — word n-grams should separate gold from topically-equivalent distractors even though unigrams (and entities, and embeddings, and cross-encoders) could not.

Same discipline as loop 6: a zero-cost separation diagnostic first. Rank gold against each query's own mined hard negatives with BM25 at increasing gram orders — arm 1 (unigram) is the control, higher arms carry the hypothesis. The pool is *maximally adversarial* for this test: the hard negatives were themselves BM25-mined, i.e. selected to out-unigram the gold. Any bigram gain here is a lower bound.

In [3]:
from src.ngram_analysis import analyze_ngram_signal

ngram_report = analyze_ngram_signal(clerc_data)  # max_n=2; sweep with max_n=5
print(ngram_report)

corpus df:   0%|          | 0/41167 [00:00<?, ?it/s]

ngram-rank:   0%|          | 0/2000 [00:00<?, ?it/s]

N-gram signal (2,000 queries, median pool 21, BM25 k1=1.5 b=0.75, gold ranked among own hard negatives):
  arm       stratum     queries  rel.rank   top-1   top-3   top-5  zero-ovl
  unigram   all           2,000     0.666    8.6%   15.2%   21.7%      0.0%
  unigram   quoted        1,458     0.662    9.1%   16.0%   22.6%      0.0%
  unigram   unquoted        542     0.676    7.2%   12.9%   19.2%      0.0%
  uni+2gram all           2,000     0.628    9.3%   16.0%   24.5%      0.0%
  uni+2gram quoted        1,458     0.625   10.2%   16.8%   25.2%      0.0%
  uni+2gram unquoted        542     0.635    7.0%   13.8%   22.7%      0.0%
  (random top-1 baseline: 4.8%; random rel.rank: 0.500)


#### Tier 0 verdict — the first hypothesis to survive

| step | Δ rel.rank | Δ top-1 | Δ top-5 |
|---|---|---|---|
| 1 → 2 grams | **−0.038** | **+0.7pt** | **+2.8pt** |
| 2 → 3 | −0.018 | −0.1 | +1.3 |
| 3 → 4 | −0.010 | +0.1 | +0.4 |
| 4 → 5 | −0.005 | 0.0 | +0.2 |

Bigrams improve every cutoff (rel.rank 0.666 → 0.628, top-5 21.7% → 24.5%), and top-1 gains concentrate in the *quoted* stratum (+1.1pt vs −0.2pt unquoted) — the mechanism fingerprint the §4.10 insight predicted.

The `max_n` sweep also falsifies the naive extension ("longer grams must be better"): **top-1 saturates at n=2**, and each further order buys half of what the previous one did. Two mechanisms explain it, both already on the record: the paper's ≥25% quote-alteration rate shatters 4- and 5-grams while leaving bigrams intact, and legal boilerplate is shared verbatim across opinions, so surviving long grams add little after IDF discounting. **Bigrams are the alteration-tolerant length.** Engineering answer: `max_n=2`.

#### Tier 1: rerank the real pool

The diagnostic pool was rigged against lexical scorers; the real question is whether the bigram signal survives contact with actual retrieval output. Same `Reranker` seam the two failed rerankers used — only the strategy changes, so the comparison isolates exactly one variable.

Success criterion, stated in advance: beat the un-reranked hybrid (**ndcg@10 = 0.2064**) with the win/loss/tie split favoring wins — unlike Cohere Rerank 4 Pro, which scored 0.13 with recall@10 *dropping below its own first stage*. Ceiling acknowledged: reranking cannot fix the ~65% of queries whose gold is absent from the pool; recall@100 ≈ 0.88 bounds everything below.

In [42]:
from src.ngram_analysis import NgramStats
from src.rerank import NgramBM25Strategy, Reranker
from src.evaluation import evaluate_retrieval

ngram_stats = NgramStats.build(
    clerc_data.corpus,
    [q.query for q in clerc_data.queries.values()],
    max_n=2,
)

searcher_bigram = Reranker(
    retriever=searcher_qwen_hybrid,
    strategy=NgramBM25Strategy(ngram_stats),
    multiplier=10,
)

corpus df:   0%|          | 0/41167 [00:00<?, ?it/s]

In [43]:
report_bigram = evaluate_retrieval(searcher_bigram, clerc_data)
print(report_bigram)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
from src.ngram_analysis import NgramStats
from src.rerank import FusedNgramStrategy, Reranker
from src.evaluation import compare_pair, RetrievalMetric

ngram_stats = NgramStats.build(
    clerc_data.corpus,
    [q.query for q in clerc_data.queries.values()],
    max_n=2,
)

searcher_v2_fused = Reranker(
    retriever=searcher_v2,
    strategy=FusedNgramStrategy(ngram_stats),   # rrf_k=60 default
    multiplier=10,
)

corpus df:   0%|          | 0/41167 [00:00<?, ?it/s]

In [22]:
report = compare_pair(
    {'hybrid-v2': searcher_v2, 'hybrid-v2+bigram-fused': searcher_v2_fused},
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10,
)
print(report)

search:hybrid-v2:   0%|          | 0/2000 [00:00<?, ?it/s]

search:hybrid-v2+bigram-fused:   0%|          | 0/2000 [00:00<?, ?it/s]

hybrid-v2+bigram-fused vs hybrid-v2 (2000 queries, top-10, 95% CI):
  NDCG@10:    Δ = +0.018   95% CI [+0.011, +0.025]   p = 8.5e-10   hybrid-v2+bigram-fused>hybrid-v2: 20.7% | hybrid-v2>hybrid-v2+bigram-fused: 10.5% | ties: 68.8%
  Recall@10:  Δ = +0.025   95% CI [+0.011, +0.039]   p = 5.9e-04   hybrid-v2+bigram-fused>hybrid-v2: 6.6% | hybrid-v2>hybrid-v2+bigram-fused: 4.0% | ties: 89.4%
  MRR@10:     Δ = +0.017   95% CI [+0.010, +0.024]   p = 4.4e-10   hybrid-v2+bigram-fused>hybrid-v2: 20.7% | hybrid-v2>hybrid-v2+bigram-fused: 10.5% | ties: 68.8%


In [23]:
report

PairwiseComparisonReport(name_a='hybrid-v2', name_b='hybrid-v2+bigram-fused', num_queries=2000, top_k=10, ci_level=0.95, n_bootstrap=10000, n_permutations=10000, metrics=[PairwiseMetric(metric=<RetrievalMetric.NDCG_10: 'ndcg@10'>, mean_a=0.17500697385183792, mean_b=0.19333558708026483, delta=0.01832861322842689, ci_low=0.011195824958611563, ci_high=0.025381430391776424, p_value=8.544143817996594e-10, win_rate=0.207, loss_rate=0.1055), PairwiseMetric(metric=<RetrievalMetric.RECALL_10: 'recall@10'>, mean_a=0.336, mean_b=0.361, delta=0.025, ci_low=0.011, ci_high=0.039, p_value=0.0005947132623332928, win_rate=0.0655, loss_rate=0.0405), PairwiseMetric(metric=<RetrievalMetric.MRR_10: 'mrr@10'>, mean_a=0.12649503968253967, mean_b=0.14305019841269842, delta=0.01655515873015873, ci_low=0.00960686507936508, ci_high=0.023551418650793646, p_value=4.363176797661243e-10, win_rate=0.207, loss_rate=0.1055)])

In [24]:
for m in report.metrics:
    print(f"{m.metric.value}:  {m.mean_a:.4f} -> {m.mean_b:.4f}  (Δ {m.delta:+.4f})")

ndcg@10:  0.1750 -> 0.1933  (Δ +0.0183)
recall@10:  0.3360 -> 0.3610  (Δ +0.0250)
mrr@10:  0.1265 -> 0.1431  (Δ +0.0166)


#### Tier 1 results: a good feature is a bad dictator

The replacement reranker was killed mid-run — by 182/2000 queries the direction was already unmistakable: ndcg ≈ 0.186 (< 0.2064) with **recall@10 ≈ 0.31, below the first stage's 0.40** — the same eviction signature as every failed reranker before it. Interestingly, mrr ticked *up*: when a quoted span fires, bigrams slam gold to #1; when nothing fires, they reshuffle by lexical noise and destroy the qwen+RRF ordering they replaced.

Diagnosis: Tier 0 validated bigrams as a *feature*; replacement reranking made them a *dictator* — discarding the first stage's signal on 100% of queries to exploit evidence decisive on ~15%. The fix follows immediately: fuse, don't replace. `FusedNgramStrategy` RRF-merges the incoming order with the bigram order, with one invariant by construction: candidates with zero n-gram evidence keep their first-stage position — **bigrams can promote, never veto**.

In [ ]:
from src.rerank import FusedNgramStrategy, Reranker
from src.evaluation import compare_pair, RetrievalMetric

searcher_v2_fused = Reranker(
    retriever=searcher_v2,
    strategy=FusedNgramStrategy(ngram_stats),   # rrf_k=60 default
    multiplier=10,
)

In [ ]:
report = compare_pair(
    {'hybrid-v2': searcher_v2, 'hybrid-v2+bigram-fused': searcher_v2_fused},
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10,
)
print(report)

The fused voter, tested directionally on the fast local pool (`searcher_v2`):

| metric | hybrid-v2 | +bigram voter | Δ |
|---|---|---|---|
| ndcg@10 | 0.1750 | **0.1933** | +0.0183 (p = 8.5e-10) |
| recall@10 | 0.3360 | **0.3610** | +0.0250 (p = 5.9e-04) |
| mrr@10 | 0.1265 | **0.1431** | +0.0166 (p = 4.4e-10) |

Wins/losses/ties: **20.7% / 10.5% / 68.8%** — the first intervention in this entire study with the healthy profile: positive on every metric, wins 2:1, and **recall@10 rising** (golds promoted from positions 11–100 into the top-10) instead of the eviction signature. For scale: 0.1933 beats Legal ModernBERT (0.1924) using ~40 lines of model-free Python — and lands within a whisker of qwen-dense (0.1978).

#### Tier 2: the indexed bigram slot

The fused reranker proved the voter transfers to real pools; its structural limits are the pool itself (it can never *add* candidates → recall@100 is untouchable) and the client-side payload tax (100 full texts per query). Tier 2 removes both by making the bigram signal a first-class vector slot.

One deployment lesson learned the hard way: this server version does **not** allow adding a sparse slot to an existing collection (`update_collection` rejects new vector names), so the collection is recreated with the full 3-slot schema. The `EmbeddingCache` makes that affordable — the 41k qwen vectors upload from disk with **zero re-paid inference**; only BM25 (server-side) and the local ngram encoding run again.

- Document values carry BM25-saturated TF; query values are 1.0; `Modifier.IDF` makes the server apply collection-wide IDF — the local IDF table retires;
- Fusion becomes native `FusionQuery(RRF)` across three prefetches — the custom client-side RRF retires too.

Three systems, one criterion (beat **0.2064** with the healthy profile):

| system | slots | question |
|---|---|---|
| (a) baseline | dense + bm25 | the bar |
| (b) replace | dense + ngram | does uni+bigram subsume stemmed BM25? |
| (c) extend | dense + bm25 + ngram | does the third voter transfer server-side? |

In [49]:
from qdrant_client.models import Modifier
from tqdm.auto import tqdm
from src.indexing import DocumentIndexer, EmbeddingConfig

# 1. Harvest the already-computed bm25 vectors from the old collection into
#    the cache — bypasses the slow cloud-inference proxy entirely.
bm25_cache = embedding_cache.load('qdrant/bm25', 'sparse')
offset = None
with tqdm(total=len(clerc_data.corpus), desc='harvest bm25') as bar:
    while True:
        records, offset = inference_client.scroll(
            COLLECTION_NAME_CLOUD,
            limit=1000,
            with_payload=False,
            with_vectors=['sparse'],
            offset=offset,
        )
        for r in records:
            bm25_cache[str(r.id)] = r.vector['sparse']
        bar.update(len(records))
        if offset is None:
            break
embedding_cache.save('qdrant/bm25', 'sparse')

# 2. All three slots now embed client-side or stream from cache:
#    dense = cached qwen, bm25 = harvested, ngram = local encoder.
bm25_local = configs_cloud[1].model_copy(update={'local_inference': True})
bigram_config = EmbeddingConfig(
    name='bm25_bigram',
    model_id='ngram-bm25-uni2',
    kind='sparse',
    backend='ngram',
    modifier=Modifier.IDF,
    doc_options={'max_n': 2, 'avg_tokens': ngram_stats.mean_tokens},
)
configs_cloud_tri = [configs_cloud[0], bm25_local, bigram_config]
COLLECTION_NAME_CLOUD_TRI = 'clerc_cloud_tri'

indexer_tri = DocumentIndexer(
    inference_client,
    COLLECTION_NAME_CLOUD_TRI,
    embeddings=configs_cloud_tri,
    cache=embedding_cache,
)
indexer_tri.ensure_collection()

harvest bm25:   0%|          | 0/41167 [00:00<?, ?it/s]

In [ ]:
indexer_tri.col

[EmbeddingConfig(name='dense', model_id='qwen/qwen3-embedding-8b', kind='dense', backend='openrouter', size=4096, distance=<Distance.COSINE: 'Cosine'>, providers=None, parallel=16, device=None, query_prompt='Instruct: Given a passage from a legal opinion with a citation removed, retrieve the cited case passage.\nQuery: ', modifier=None, hnsw_m=None),
 EmbeddingConfig(name='sparse', model_id='qdrant/bm25', kind='sparse', backend='fastembed', size=None, distance=<Distance.COSINE: 'Cosine'>, providers=None, parallel=None, device=None, query_prompt=None, modifier=<Modifier.IDF: 'idf'>, hnsw_m=None),
 EmbeddingConfig(name='bm25_bigram', model_id='ngram-bm25-uni2', kind='sparse', backend='ngram', size=None, distance=<Distance.COSINE: 'Cosine'>, providers=None, parallel=None, device=None, query_prompt=None, modifier=<Modifier.IDF: 'idf'>, hnsw_m=None)]

In [55]:
# dense slot streams from the embedding cache — no OpenRouter re-spend
indexer_tri.upload(
    items=list(clerc_data.corpus.values()),
    batch_size=32,
    skip_existing=True,
    parallel=10,
)

upsert:   0%|          | 0/2696 [00:00<?, ?it/s]

In [ ]:
from src.search import HybridSearcher
from src.evaluation import evaluate_retrieval

searcher_qwen_ngram = HybridSearcher(          # (b) replace bm25 with ngram
    inference_client, COLLECTION_NAME_CLOUD_TRI, [configs_cloud[0], bigram_config]
)
searcher_qwen_tri = HybridSearcher(            # (c) all three voters
    inference_client, COLLECTION_NAME_CLOUD_TRI, configs_cloud_tri
)
searcher_qwen_tri.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)


openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/32 [00:00<?, ?it/s]

In [63]:
searcher_qwen_ngram.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)

openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/29 [00:00<?, ?it/s]

In [64]:
searcher_qwen_ngram_report = evaluate_retrieval(searcher_qwen_ngram, clerc_data)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
print(searcher_qwen_ngram_report)

In [ ]:
searcher_qwen__tri_ngram_report = evaluate_retrieval(searcher_qwen_tri, clerc_data)

search:   0%|          | 0/200 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [60]:
searcher_qwen__tri_ngram_report = evaluate_retrieval(searcher_qwen_tri, clerc_data)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

In [59]:
print(searcher_qwen__tri_ngram_report)

Retrieval evaluation (200 queries, top-10):
  ndcg@10      0.2148
  recall@10    0.3850
  mrr@10       0.1633


In [61]:
print(searcher_qwen__tri_ngram_report)

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.2133
  recall@10    0.3970
  mrr@10       0.1581


In [68]:
from src.evaluation import compare_pair, RetrievalMetric

compare_pair(
    {'searcher_qwen_ngram': searcher_qwen_ngram, 'searcher_qwen_tri': searcher_qwen_tri},
    data=clerc_data,
    metrics=[RetrievalMetric.NDCG_10, RetrievalMetric.RECALL_10, RetrievalMetric.MRR_10],
    top_k=10,
)

search:searcher_qwen_ngram:   0%|          | 0/2000 [00:00<?, ?it/s]

search:searcher_qwen_tri:   0%|          | 0/2000 [00:00<?, ?it/s]

PairwiseComparisonReport(name_a='searcher_qwen_ngram', name_b='searcher_qwen_tri', num_queries=2000, top_k=10, ci_level=0.95, n_bootstrap=10000, n_permutations=10000, metrics=[PairwiseMetric(metric=<RetrievalMetric.NDCG_10: 'ndcg@10'>, mean_a=0.21633267957707888, mean_b=0.2130991789659301, delta=-0.0032335006111488132, ci_low=-0.008863090160224926, ci_high=0.0024162733576771084, p_value=0.0390941835172061, win_rate=0.1265, loss_rate=0.1775), PairwiseMetric(metric=<RetrievalMetric.RECALL_10: 'recall@10'>, mean_a=0.4105, mean_b=0.397, delta=-0.0135, ci_low=-0.0245, ci_high=-0.0025, p_value=0.01744380153603954, win_rate=0.0255, loss_rate=0.039), PairwiseMetric(metric=<RetrievalMetric.MRR_10: 'mrr@10'>, mean_a=0.15794940476190475, mean_b=0.1578390873015873, delta=-0.00011031746031745971, ci_low=-0.006101726190476189, ci_high=0.00590927579365079, p_value=0.07117867624190494, win_rate=0.1265, loss_rate=0.1775)])

#### Loop 7 verdict


The Tier-2 matrix, 2000 queries, top-10:

| System          | Slots                      | NDCG@10 | Recall@10 | MRR@10 |
|-----------------|----------------------------|--------:|----------:|-------:|
| (a) baseline    | qwen + BM25                | 0.2064  | 0.4000    | 0.1450 |
| (b) challenger  | qwen + n-gram BM25 (n≤2)   | 0.2163  | 0.4105    | 0.1580 |
| (c) challenger  | qwen + BM25 + n-gram BM25  | 0.2133  | 0.3970    | 0.1580 |

The bigram slot wins, and adding the unigram slot back provides no benefit — it measurably hurts. The mechanism is not mysterious: the n-gram encoder with n ≤ 2 already emits unigrams alongside bigrams, so slot (b) contains everything BM25 knows. RRF voters must be information-diverse; a redundant voter is negative weight.


Recommendation, however, is not (b). The matrix above optimizes the NFRs we instrumented — ranking quality under paired significance. It says nothing about the NFRs we did not test but a production system must answer for: query latency, embedding cost, and operational footprint. The qwen dense leg requires an external embedding API on every query and corpus-scale re-embedding on every index rebuild. The evidence from Loop 7's Tier 1.5 says the bigram signal does not need it: local dense + bigram fusion reached 0.1933 (from 0.1750, p = 8.5e-10, wins 20.7% vs 10.5%, recall up) — beating the domain-tuned Legal ModernBERT (0.1924) with zero external dependencies.

**The recommended production configuration is therefore the fully local hybrid: simple dense embeddings + bigram BM25 sparse — roughly 90% of the champion's quality at a fraction of its cost and none of its API surface.** The champion (b) is what you deploy when the extra ~0.02 ndcg is worth an embedding API in the hot path; the eval gives you the number to make that trade explicitly rather than by default.

### 4.12 Closing the loop: the satisfaction gate

What does the satisfaction gate say now? Re-read the decomposition in §2.3: search's consumer is **RAG — a programmatic consumer**. The junior associate never sees the top-10; the LLM does. So top-10 *precision* was never the binding requirement. **recall@K is** — and recall@100 is already **0.88**.

The correct next move is therefore architectural, not algorithmic: **widen K, hand more candidates to the generation stage, and let the LLM — which *can* read for verbatim spans — pick the citation.**. This also means that the burden of the evaluation switches to the other component. We might need end-to-end testing in the end to be sure both components work as intended. 

There are multiple modes of testing joint components, and that is a topic for a separate article. The current article showed that the evaluation target moves as our understand of domain deepers: from `ndcg@10` we move to `recall@100`. However now we get a new quantity we need to validate: that out of these 100 elements LLM selects the most relevant. **Each simplification hides a tradeoff behind its simplicity**

## Final Conclusion 

![Abstracted methodology](./diagrams/evaluation-methodology-abstract.svg)



You can find final methodology under the following [link](https://app.diagrams.net/#G1B7syKvDVwjy4FM8kAfEypLwthDMlO8sk#%7B%22pageId%22%3A%22ekK9FnIo51W-imU3G7XK%22%7D). The most important lesson is of course to remember that no methodology can guarantee 100% of the coverage, evaluation as other disciplines of the SE are non-deterministic by nature. All we can do is to gather best practices and build a path towards our success.

In [69]:
from src.ngram_analysis import NgramSparseEncoder


encoder = NgramSparseEncoder()
encoder.encode_query("Hello there I")

SparseVector(indices=[2540066372, 907060870, 3865851505, 1513089430, 3059329015], values=[1.0, 1.0, 1.0, 1.0, 1.0])

In [70]:
from src.ngram_analysis import NgramSparseEncoder


encoder = NgramSparseEncoder(1)
encoder.encode_query("Hello there I")

SparseVector(indices=[3865851505, 1513089430, 907060870], values=[1.0, 1.0, 1.0])